In [ ]:
# Run from the repository root so that relative result paths resolve
# identically regardless of where this notebook lives.
from oddball_eeg.paths import use_project_root

PROJECT_ROOT = use_project_root()

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# Load current config
# ============================================================

from oddball_eeg.config import get_experiment_family

print("Project root:")
print(PROJECT_ROOT)

In [ ]:
EXPERIMENT_NAMES = [
    "ds003061_within_subject_max30_wide_timebin",
    "ds005863_within_subject_max30_wide_timebin",
    "ds006466_within_subject_max30_wide_timebin",
    "ds006480_within_subject_max30_wide_timebin",
]


def dataset_label_from_cfg(cfg) -> str:
    if cfg.dataset.dataset == "ds006480":
        return f"{cfg.dataset.dataset}_{cfg.dataset.event_preset}"

    return cfg.dataset.dataset


def normalize_subject_id(value, dataset_label: str | None = None) -> str:
    """
    Same normalization logic used in Notebook 27.
    """
    if pd.isna(value):
        return np.nan

    subject = str(value).strip()

    if subject.startswith("sub-"):
        subject = subject[4:]

    if subject.endswith(".0"):
        subject = subject[:-2]

    label = str(dataset_label or "")

    if subject.isdigit():

        if label.startswith(("ds003061", "ds005863")):
            return subject.zfill(3)

        if label.startswith("ds006480"):
            return subject.zfill(2)

    return subject

In [ ]:
for experiment_name in EXPERIMENT_NAMES:

    family = get_experiment_family(experiment_name)
    cfg_loso = family["loso"]

    print("=" * 100)
    print("Experiment :", experiment_name)
    print("Dataset    :", dataset_label_from_cfg(cfg_loso))
    print("Subjects   :", len(cfg_loso.subjects.subjects))
    print("Seeds      :", cfg_loso.seeds.seeds)
    print()
    print("LOSO root:")
    print(cfg_loso.result_dir())

    for seed in cfg_loso.seeds.seeds:

        fold_path = (
            cfg_loso.result_dir(seed=seed)
            / "tables"
            / "folds"
            / "loso_fold_results.csv"
        )

        history_dir = (
            cfg_loso.result_dir(seed=seed)
            / "histories"
            / "eegnet"
        )

        print()
        print(f"Seed {seed}")
        print("  Fold results :", fold_path)
        print("  Histories    :", history_dir)

In [ ]:
records = []

for experiment_name in EXPERIMENT_NAMES:

    family = get_experiment_family(experiment_name)
    cfg_loso = family["loso"]

    dataset_label = dataset_label_from_cfg(cfg_loso)

    for seed in cfg_loso.seeds.seeds:

        # ----------------------------------------------------
        # Exact fold-results file written by Notebook 04
        # ----------------------------------------------------

        fold_results_path = (
            cfg_loso.result_dir(seed=seed)
            / "tables"
            / "folds"
            / "loso_fold_results.csv"
        )

        if not fold_results_path.exists():
            raise FileNotFoundError(
                f"Missing final LOSO fold-results file:\n"
                f"{fold_results_path}"
            )

        fold_df = pd.read_csv(fold_results_path)

        if "model" not in fold_df.columns:
            raise ValueError(
                f"'model' column missing from:\n"
                f"{fold_results_path}"
            )

        eegnet_df = fold_df[
            fold_df["model"]
            .astype(str)
            .str.lower()
            .eq("eegnet")
        ].copy()

        if eegnet_df.empty:
            raise RuntimeError(
                f"No EEGNet rows found in:\n"
                f"{fold_results_path}"
            )

        # ----------------------------------------------------
        # Exact history directory written by Notebook 04
        # ----------------------------------------------------

        history_dir = (
            cfg_loso.result_dir(seed=seed)
            / "histories"
            / "eegnet"
        )

        if not history_dir.exists():
            raise FileNotFoundError(
                f"Missing EEGNet history directory:\n"
                f"{history_dir}"
            )

        # ----------------------------------------------------
        # Map saved history filenames to normalized subjects
        # ----------------------------------------------------

        history_map = {}

        for history_path in history_dir.glob(
            "eegnet_history_heldout-subject-*.csv"
        ):

            subject_token = (
                history_path.stem
                .replace(
                    "eegnet_history_heldout-subject-",
                    "",
                )
            )

            subject_norm = normalize_subject_id(
                subject_token,
                dataset_label,
            )

            history_map[subject_norm] = history_path

        # ----------------------------------------------------
        # Match every EEGNet result row to its actual history
        # ----------------------------------------------------

        for _, row in eegnet_df.iterrows():

            original_subject = row.get(
                "heldout_subject",
                row.get("subject"),
            )

            subject = normalize_subject_id(
                original_subject,
                dataset_label,
            )

            history_path = history_map.get(subject)

            if history_path is None:
                raise FileNotFoundError(
                    "Could not match EEGNet result to history.\n"
                    f"Dataset: {dataset_label}\n"
                    f"Subject: {subject}\n"
                    f"Seed: {seed}\n"
                    f"History directory: {history_dir}"
                )

            history_df = pd.read_csv(history_path)

            required_history_columns = {
                "epoch",
                "train_loss",
                "val_loss",
                "val_auc",
            }

            missing = (
                required_history_columns
                - set(history_df.columns)
            )

            if missing:
                raise ValueError(
                    f"History missing columns {sorted(missing)}:\n"
                    f"{history_path}"
                )

            records.append({
                "experiment_name": experiment_name,
                "dataset": cfg_loso.dataset.dataset,
                "dataset_label": dataset_label,
                "subject": subject,
                "seed": int(seed),

                # IMPORTANT:
                # use the actual saved checkpoint information
                # from Notebook 04 rather than recomputing it.
                "best_epoch": int(row["best_epoch"]),
                "best_val_auc": float(row["best_val_auc"]),

                "test_auc": float(row["auc"]),

                "n_training_epochs_completed": int(
                    len(history_df)
                ),

                "history_path": str(history_path),
                "fold_results_path": str(
                    fold_results_path
                ),
            })


audit = pd.DataFrame(records)

print("Loaded LOSO EEGNet runs:", len(audit))

audit.head()

In [ ]:
print("=" * 90)
print("FINAL LOSO EEGNET AUDIT")
print("=" * 90)

print()
print("Runs by dataset:")
print(
    audit
    .groupby("dataset_label")
    .size()
)

print()
print("Seeds:")
print(
    sorted(audit["seed"].unique())
)

print()
print("Total runs:")
print(len(audit))

print()
print("Median selected epoch:")
print(audit["best_epoch"].median())

print()
print("Median completed epochs:")
print(audit["n_training_epochs_completed"].median())


# ------------------------------------------------------------
# Final-thesis expectations
# ------------------------------------------------------------

expected_counts = {
    "ds003061": 13 * 3,
    "ds005863": 30 * 3,
    "ds006466": 30 * 3,
    "ds006480_pooled": 30 * 3,
}

actual_counts = (
    audit
    .groupby("dataset_label")
    .size()
    .to_dict()
)

print()
print("Expected counts:")
print(expected_counts)

print()
print("Actual counts:")
print(actual_counts)


if len(audit) != 309:
    raise RuntimeError(
        f"Expected 309 final LOSO EEGNet runs, "
        f"but loaded {len(audit)}."
    )

if actual_counts != expected_counts:
    raise RuntimeError(
        "Dataset run counts do not match the final thesis benchmark."
    )

if float(audit["best_epoch"].median()) != 38.0:
    raise RuntimeError(
        "The median best epoch is not 38. "
        "This strongly suggests the wrong result family was loaded."
    )

if float(audit["n_training_epochs_completed"].median()) != 48.0:
    raise RuntimeError(
        "The median completed training length is not 48. "
        "Check that these are the final LOSO histories."
    )

print()
print("PASS: These match the final 309-run LOSO benchmark.")

In [ ]:
median_best_epoch = float(
    audit["best_epoch"].median()
)

median_completed = float(
    audit["n_training_epochs_completed"].median()
)

median_val_auc = float(
    audit["best_val_auc"].median()
)


selection = audit.copy()

selection["distance_best_epoch"] = (
    selection["best_epoch"]
    - median_best_epoch
).abs()

selection["distance_completed"] = (
    selection["n_training_epochs_completed"]
    - median_completed
).abs()

selection["distance_val_auc"] = (
    selection["best_val_auc"]
    - median_val_auc
).abs()


representative = (
    selection
    .sort_values(
        [
            "distance_best_epoch",
            "distance_completed",
            "distance_val_auc",
            "dataset_label",
            "subject",
            "seed",
        ]
    )
    .iloc[0]
)


print("=" * 90)
print("REPRESENTATIVE LOSO RUN")
print("=" * 90)

print("Dataset             :", representative["dataset_label"])
print("Subject             :", representative["subject"])
print("Seed                :", representative["seed"])
print("Best epoch          :", representative["best_epoch"])
print(
    "Training completed  :",
    representative["n_training_epochs_completed"],
)
print(
    "Best validation AUC :",
    f"{representative['best_val_auc']:.4f}",
)

print()
print("History:")
print(representative["history_path"])

print()
print("Fold-results source:")
print(representative["fold_results_path"])

In [ ]:
history_path = Path(
    representative["history_path"]
)

history = pd.read_csv(history_path)

history = history[
    [
        "epoch",
        "train_loss",
        "val_loss",
        "val_auc",
    ]
].copy()

selected_epoch = int(
    representative["best_epoch"]
)

print(history.head())

print()
print(history.tail())

print()
print("Selected checkpoint:", selected_epoch)

In [ ]:
# ============================================================
# Two-panel representative EEGNet training diagnostic
# Left: training and validation loss
# Right: validation ROC-AUC
# ============================================================

OUTPUT_DIR = PROJECT_ROOT / "thesis_figures_final"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PNG_PATH = (
    OUTPUT_DIR
    / "fig05_03_representative_training_validation_diagnostics.png"
)

PDF_PATH = (
    OUTPUT_DIR
    / "fig05_03_representative_training_validation_diagnostics.pdf"
)


# ------------------------------------------------------------
# Selected checkpoint information
# ------------------------------------------------------------

selected_epoch = int(representative["best_epoch"])
best_val_auc = float(representative["best_val_auc"])

selected_row = history.loc[
    history["epoch"] == selected_epoch
].iloc[0]

selected_train_loss = float(selected_row["train_loss"])
selected_val_loss = float(selected_row["val_loss"])
selected_val_auc = float(selected_row["val_auc"])


print("=" * 80)
print("REPRESENTATIVE RUN USED FOR FIGURE")
print("=" * 80)
print("Dataset:", representative["dataset_label"])
print("Subject:", representative["subject"])
print("Seed:", representative["seed"])
print("Selected checkpoint:", selected_epoch)
print(f"Training loss at checkpoint:   {selected_train_loss:.4f}")
print(f"Validation loss at checkpoint: {selected_val_loss:.4f}")
print(f"Validation AUC at checkpoint:  {selected_val_auc:.4f}")
print(f"Saved best validation AUC:     {best_val_auc:.4f}")


# ------------------------------------------------------------
# Create two-panel figure
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12.0, 4.8),
    sharex=True,
)

ax_loss = axes[0]
ax_auc = axes[1]


# ============================================================
# Panel A: training and validation loss
# ============================================================

ax_loss.plot(
    history["epoch"],
    history["train_loss"],
    linewidth=2.0,
    label="Training loss (class-weighted)",
)

ax_loss.plot(
    history["epoch"],
    history["val_loss"],
    linewidth=2.0,
    label="Validation loss (unweighted)",
)

ax_loss.axvline(
    selected_epoch,
    linestyle="--",
    linewidth=1.5,
    label=f"Selected checkpoint (epoch {selected_epoch})",
)

ax_loss.scatter(
    [selected_epoch],
    [selected_train_loss],
    s=45,
    zorder=5,
)

ax_loss.scatter(
    [selected_epoch],
    [selected_val_loss],
    s=45,
    zorder=5,
)

ax_loss.set_xlabel(
    "Training epoch",
    fontsize=12,
)

ax_loss.set_ylabel(
    "Cross-entropy loss",
    fontsize=12,
)

ax_loss.set_title(
    "(a) Training and validation loss",
    fontsize=12,
)

ax_loss.tick_params(
    axis="both",
    labelsize=10.5,
)

ax_loss.grid(
    True,
    alpha=0.25,
)

ax_loss.legend(
    fontsize=9.5,
    frameon=False,
    loc="best",
)


# ============================================================
# Panel B: validation ROC-AUC
# ============================================================

ax_auc.plot(
    history["epoch"],
    history["val_auc"],
    linewidth=2.0,
    label="Validation ROC-AUC",
)

ax_auc.axvline(
    selected_epoch,
    linestyle="--",
    linewidth=1.5,
    label=f"Selected checkpoint (epoch {selected_epoch})",
)

ax_auc.scatter(
    [selected_epoch],
    [selected_val_auc],
    s=55,
    zorder=5,
    label=f"Best validation AUC = {selected_val_auc:.3f}",
)

ax_auc.set_xlabel(
    "Training epoch",
    fontsize=12,
)

ax_auc.set_ylabel(
    "Validation ROC-AUC",
    fontsize=12,
)

ax_auc.set_title(
    "(b) Validation ROC-AUC",
    fontsize=12,
)

ax_auc.tick_params(
    axis="both",
    labelsize=10.5,
)

ax_auc.grid(
    True,
    alpha=0.25,
)

ax_auc.legend(
    fontsize=9.5,
    frameon=False,
    loc="best",
)


# ------------------------------------------------------------
# Common formatting
# ------------------------------------------------------------

fig.tight_layout(
    w_pad=2.5
)


# ------------------------------------------------------------
# Save high-resolution thesis outputs
# ------------------------------------------------------------

fig.savefig(
    PNG_PATH,
    dpi=300,
    bbox_inches="tight",
)

fig.savefig(
    PDF_PATH,
    bbox_inches="tight",
)

plt.show()


print()
print("Saved figure:")
print(PNG_PATH)
print(PDF_PATH)

In [ ]:
PROVENANCE_PATH = (
    OUTPUT_DIR
    / "fig05_03_representative_training_validation_loss_provenance.csv"
)

SOURCE_DATA_PATH = (
    OUTPUT_DIR
    / "fig05_03_representative_training_validation_loss_data.csv"
)


history.to_csv(
    SOURCE_DATA_PATH,
    index=False,
)


pd.DataFrame(
    [{
        "selection_rule": (
            "Closest to the median LOSO selected epoch, "
            "then median completed training length, "
            "then median best validation AUC; "
            "outer-test AUC was not used for selection."
        ),

        "total_final_loso_eegnet_runs": len(audit),

        "overall_median_best_epoch":
            median_best_epoch,

        "overall_median_completed_epochs":
            median_completed,

        "overall_median_best_validation_auc":
            median_val_auc,

        "dataset":
            representative["dataset_label"],

        "subject":
            representative["subject"],

        "seed":
            int(representative["seed"]),

        "selected_checkpoint":
            int(representative["best_epoch"]),

        "completed_epochs":
            int(
                representative[
                    "n_training_epochs_completed"
                ]
            ),

        "best_validation_auc":
            float(
                representative["best_val_auc"]
            ),

        "source_history_csv":
            representative["history_path"],

        "source_fold_results_csv":
            representative["fold_results_path"],
    }]
).to_csv(
    PROVENANCE_PATH,
    index=False,
)


print("Source data:")
print(SOURCE_DATA_PATH)

print()
print("Provenance:")
print(PROVENANCE_PATH)